In [1]:
# === Core Python ===
import os
import glob
import collections
import cftime
from datetime import datetime
from typing import Tuple, Dict, Optional

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs
from xskillscore import rmse, pearson_r

# === Scientific Computation ===
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt, lfilter
from scipy.optimize import curve_fit
from skimage.feature import peak_local_max
import metpy.calc as mpcalc  # Meteorology/thermodynamics

# === Plotting & Visualization ===
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Mapping & Cartography ===
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# === Visualization Helpers ===
import geocat.viz.util as gvutil
import geocat.viz as gv
import cmaps as gvcmaps  # Optional, check if available
import warnings


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
class ObsDataReader:
    """
    Class to read and preprocess observational data with internal dataset registry.
    Supports regional mean extraction.
    """
    
    def __init__(self):
        self.obs_group = self._define_obs_group()

    def _define_obs_group(self):
        return {
            'ERA5': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/monthly',
                    'template': 'ERA5_analysis_monthly_%(year).nc',
                    'period': '2001-2018'
                },
                '6hourly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/6hourly',
                    'template': 'ERA5.6hourly.en00.%(var).%(year)01-%(year)12.nc',
                    'period': '1979-2018'
                }
            },
            'GPCP': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/monthly',
                    'template': 'PRECT.monthly.%(year).nc',
                    'period': '1979-2017'
                },
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            },
            'IMERG': {
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/IMERG/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            }
        }

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-10, 10), (-90, 60)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    def get_group(self, name=None):
        return self.obs_group.get(name) if name else self.obs_group

    def list_datasets(self):
        return list(self.obs_group.keys())

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert time to datetime64 if cftime is used."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    def read(self, var, obsname, time, frequency=None, regnam='global', regional_mean=False):
        obs_group = self.obs_group.get(obsname)
        if obs_group is None:
            raise ValueError(f"Observational dataset '{obsname}' not found.")
        if frequency not in obs_group:
            raise ValueError(f"Frequency '{frequency}' not found in '{obsname}'")

        group_info = obs_group[frequency]
        time_index = pd.to_datetime(time)
        years = sorted(set(str(y) for y in time_index.year))
        
        paths = []
        for y in years:
            fname = group_info["template"].replace("%(year)", y).replace("%(var)", var)
            paths.append(os.path.join(group_info["path"], fname))

        refds = xr.open_mfdataset(paths, decode_times=True, combine='by_coords')
        if refds.lon.min() >= 0:
            refds = refds.assign_coords(lon=((refds.lon + 180) % 360 - 180)).sortby("lon")

        if var not in refds:
            raise KeyError(f"Variable '{var}' not found in file(s): {paths}")

        data = self.convert_obs_units(var, refds[var])
        data = self._ensure_datetime64(data)

        (lat1, lat2), (lon1, lon2) = self.define_region(regnam)
        data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))

        # Nearest time match within 1-day tolerance
        data = data.sel(time=time, method='nearest', tolerance=np.timedelta64(1, 'D'))

        if regional_mean:
            weights = np.cos(np.deg2rad(data.lat))
            weights /= weights.mean()
            data = (data * weights).mean(dim=["lat", "lon"])

        return data.chunk({"time": -1}) if 'time' in data.dims else data

    def convert_obs_units(self, var, data_array):
        conversions = {
            'PRECT': lambda x: x * 86400.0 * 1000.0,
            'TREFHT': lambda x: x - 273.15,
            'TS': lambda x: x - 273.15,
            'T850': lambda x: x - 273.15,
            'PSL': lambda x: x / 100.0,
            'PS': lambda x: x / 100.0,
        }

        if 'time' not in data_array.dims:
            raise ValueError(f"Expected 'time' dimension in data for '{var}'")

        return conversions[var](data_array) if var in conversions else data_array


In [3]:
class ModelDataReader:
    def __init__(self, base_path, exp_base, regnam, frequency, component="atm"):
        self.base_path = base_path
        self.exp_base = exp_base
        self.regnam = regnam
        self.frequency = frequency
        self.component = component
        self.region = self.define_region(regnam)
        self.exp_dict = self.extract_exp_info(exp_base, base_path)

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert cftime.datetime to datetime64[ns] if needed."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    @staticmethod
    def define_region(regnam: str = 'global') -> Tuple[Tuple[float, float], Tuple[float, float]]:
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-10, 10), (-90, 60)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    @staticmethod
    def extract_exp_info(base_name, data_path) -> dict:
        exps = {
            'CTRLEN10':       {'name': 'ctrl_en10', 'nens': 10, 'period': '201112'},
            'DARTEN10':       {'name': 'dart_en10', 'nens': 10, 'period': '201112'},
            'DARTEN20':       {'name': 'dart_en20', 'nens': 20, 'period': '201112'},
            'DARTEN40':       {'name': 'dart_en40', 'nens': 40, 'period': '201112'},
            'CAPTEN10_15day': {'name': 'capt_en10', 'nens': 10, 'period': '201201-201202'},
            'DARTEN20_15day': {'name': 'dart_en20', 'nens': 20, 'period': '201201-201202'},
            'DARTEN40_15day': {'name': 'dart_en40', 'nens': 40, 'period': '201201-201202'},
        }
        return {
            exp_name: {
                'run': f"{exp_name}_{base_name}" if base_name else exp_name,
                'key': exp_data['name'],
                'nens': exp_data['nens'],
                'atm': 'archive/post/atm/180x360_aave',
                'lnd': 'archive/post/lnd/180x360_aave',
                'period': exp_data['period']
            }
            for exp_name, exp_data in sorted(exps.items())
        }

    def _convert_model_units(self, var, data):
        if var == 'PRECT':
            return data * 86400.0 * 1000.0  # m/s to mm/day
        elif var in ['TREFHT', 'TS', 'T850']:
            return data - 273.15  # K to C
        elif var in ['PSL', 'PS']:
            return data / 100.0  # Pa to hPa
        return data

    def _read_variable_single_experiment(self, var: str, model_name: str, **kwargs) -> xr.DataArray:
        return self.read_variable(var, model_name, **kwargs)

    def read_variable(self, var, exp, frequency=None, regional_mean=False):
        """
        Load and process a model variable, optionally computing regional mean.
        """
        run = self.exp_dict[exp]['run']
        component_path = self.exp_dict[exp][self.component]
        nens = self.exp_dict[exp]['nens']
        period_raw = self.exp_dict[exp]['period']
        freq = frequency or self.frequency

        if freq in ['6hourly', 'daily']:
            prefix = 'ts/daily'
            period = period_raw.split("-")[0][:4]
        elif freq == 'monthly':
            prefix = 'clim/monthly'
            period = period_raw.split("-")[0][:4] + "*"
        else:
            raise ValueError(f"Unsupported frequency: {freq}")

        (lat1, lat2), (lon1, lon2) = self.region
        ensemble_data = []

        for i in range(1, nens + 1):
            member = f"EN{i:02d}"
            search_dir = os.path.join(self.base_path, run, component_path, prefix)
            file_pattern = f"{var}.{member}.{period}.nc"
            search_path = os.path.join(search_dir, file_pattern)
            rpath = sorted(glob.glob(search_path))

            if not rpath:
                raise FileNotFoundError(f"No files for {member}: {search_path}")

            ds = xcd.open_mfdataset(
                rpath,
                combine='nested',
                concat_dim='time',
                coords='minimal',
                compat='override'
            )

            if ds.lon.min() >= 0:
                ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby("lon")

            if var not in ds:
                raise KeyError(f"Variable '{var}' missing in {rpath[0]}")

            data = self._convert_model_units(var, ds[var])
            data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))

            if regional_mean:
                weights = np.cos(np.deg2rad(data.lat))
                weights /= weights.mean()
                data = (data * weights).mean(dim=["lat", "lon"])

            data = self._ensure_datetime64(data)
            ensemble_data.append(data.expand_dims(member=[i]))

        combined = xr.concat(ensemble_data, dim="member")

        if regional_mean:
            return combined.chunk({"member": -1, "time": -1})
        else:
            return combined.chunk({"member": -1, "time": -1, "lat": -1, "lon": -1})

    def read_variable_across_experiments(self, var: str, model_list: list, **kwargs) -> Dict[str, xr.DataArray]:
        data_dict = {}
        for model_name in model_list:
            da = self._read_variable_single_experiment(var, model_name, **kwargs)
            data_dict[model_name] = da
        return data_dict


In [4]:
class S2SPredictabilityAnalyzer:
    def __init__(self, base_path, exp_base, output_dir, fig_dir, region, region_alias, var_dict,
                 model_list, obsname, frequency, component="atm"):
        self.base_path = base_path
        self.exp_base = exp_base
        self.output_dir = output_dir
        self.fig_dir = fig_dir
        self.region = region
        self.region_alias = region_alias
        self.var_dict = var_dict
        self.model_list = model_list
        self.obsname = obsname
        self.frequency = frequency
        self.component = component

        self.variable = list(var_dict.keys())[0]
        self.unit = var_dict[self.variable]['unit']
        self.alias = var_dict[self.variable]['alias']
        self.forecast_start = np.datetime64("2012-01-01")
        self.forecast_days = 60  # 2-month forecasts

    def _compute_rmse(self, ref, model):
        return float(np.sqrt(((model - ref) ** 2).mean().compute().item()))

    def _compute_corr(self, ref, model):
        corr = xr.corr(ref, model, dim='time')
        return float(corr.mean().compute().item())

    def _spatial_rmse(self, model: xr.DataArray, obs: xr.DataArray) -> xr.DataArray:
        err = (model - obs) ** 2
        weights = np.cos(np.deg2rad(model.lat))
        weights /= weights.sum()
        rmse = (err * weights).mean(dim=["lat", "lon"])
        return np.sqrt(rmse)

    def _spatial_acc(self, model: xr.DataArray, obs: xr.DataArray) -> xr.DataArray:
        model_anom = model - model.mean(dim=["lat", "lon"])
        obs_anom = obs - obs.mean(dim=["lat", "lon"])
        weights = np.cos(np.deg2rad(model.lat))
        weights /= weights.sum()

        cov = ((model_anom * obs_anom) * weights).mean(dim=["lat", "lon"])
        model_std = np.sqrt(((model_anom ** 2) * weights).mean(dim=["lat", "lon"]))
        obs_std = np.sqrt(((obs_anom ** 2) * weights).mean(dim=["lat", "lon"]))
        return cov / (model_std * obs_std)

    def _ensure_time_alignment(self):
        common_time = pd.date_range(
            start=self.forecast_start,
            periods=self.forecast_days,
            freq='D' if self.frequency == 'daily' else '6h'
        )

        try:
            self.obs_data = self.obs_data.sel(time=common_time, method='nearest', tolerance=pd.Timedelta("3h"))
        except KeyError:
            print("[WARN] Falling back to interpolation for obs_data")
            self.obs_data = self.obs_data.interp(time=common_time)

        for exp in self.model_data:
            try:
                self.model_data[exp] = self.model_data[exp].sel(time=common_time, method='nearest', tolerance=pd.Timedelta("3h"))
            except KeyError:
                print(f"[WARN] Falling back to interpolation for model {exp}")
                self.model_data[exp] = self.model_data[exp].interp(time=common_time)

    def read_data(self):
        reader = ModelDataReader(
            base_path=self.base_path,
            exp_base=self.exp_base,
            regnam=self.region,
            frequency=self.frequency,
            component=self.component
        )
        self.model_data = reader.read_variable_across_experiments(
            self.variable, self.model_list, frequency=self.frequency, regional_mean=False
        )

        obs_reader = ObsDataReader()
        common_time = pd.date_range(
            start=self.forecast_start,
            periods=self.forecast_days,
            freq='D' if self.frequency == 'daily' else '6h'
        )
        self.obs_data = obs_reader.read(
            var=self.variable,
            obsname=self.obsname,
            time=common_time,
            frequency=self.frequency,
            regnam=self.region,
            regional_mean=False
        )

        if self.frequency == "daily":
            self.obs_data['time'] = self.obs_data.indexes['time'].normalize()
            for exp in self.model_data:
                self.model_data[exp]['time'] = self.model_data[exp].indexes['time'].normalize()
        elif self.frequency == "6h":
            def round_to_6h(tindex):
                return pd.to_datetime((tindex.view('int64') // (6 * 3600 * 10**9)) * (6 * 3600 * 10**9))

            self.obs_data['time'] = round_to_6h(self.obs_data.indexes['time'])
            for exp in self.model_data:
                self.model_data[exp]['time'] = round_to_6h(self.model_data[exp].indexes['time'])

    def compute_metrics(self):
        self._ensure_time_alignment()
        metrics = {}
        for exp, da in self.model_data.items():
            ens_dim = next((d for d in da.dims if d not in ['time', 'lat', 'lon']), None)
            if ens_dim:
                rmse_vals = [self._compute_rmse(self.obs_data, da.isel({ens_dim: i})) for i in range(da.sizes[ens_dim])]
                corr_vals = [self._compute_corr(self.obs_data, da.isel({ens_dim: i})) for i in range(da.sizes[ens_dim])]
                metrics[exp] = {
                    'RMSE_mean': float(np.mean(rmse_vals)),
                    'RMSE_std': float(np.std(rmse_vals)),
                    'CORR_mean': float(np.mean(corr_vals)),
                    'CORR_std': float(np.std(corr_vals))
                }
            else:
                metrics[exp] = {
                    'RMSE': self._compute_rmse(self.obs_data, da),
                    'CORR': self._compute_corr(self.obs_data, da)
                }
        return metrics

    def plot_time_evolving_metrics(self, fontz=14, force=False):
        self._ensure_time_alignment()
        time = self.obs_data.time.values

        rmse_dict = {}
        acc_dict = {}

        for i, (exp, model) in enumerate(self.model_data.items()):
            rmse_file = os.path.join(self.output_dir, f"time_rmse_{exp}.nc")
            acc_file = os.path.join(self.output_dir, f"time_acc_{exp}.nc")

            if os.path.exists(rmse_file) and os.path.exists(acc_file) and not force:
                print(f"[INFO] Skipping computation for {exp}, files exist. Loading from file.")
                rmse_all = xr.open_dataset(rmse_file)["rmse"]
                acc_all = xr.open_dataset(acc_file)["acc"]
            else:
                ens_dim = next((d for d in model.dims if d not in ['time', 'lat', 'lon']), None)

                if ens_dim:
                    rmse_list = []
                    acc_list = []
                    for j in range(model.sizes[ens_dim]):
                        member = model.isel({ens_dim: j})
                        rmse = self._spatial_rmse(member, self.obs_data)
                        acc_t = [self._spatial_acc(member.isel(time=t), self.obs_data.isel(time=t)) for t in range(member.sizes['time'])]
                        acc = xr.concat(acc_t, dim='time')
                        rmse_list.append(rmse)
                        acc_list.append(acc)
                    rmse_all = xr.concat(rmse_list, dim='member')
                    acc_all = xr.concat(acc_list, dim='member')
                else:
                    rmse_all = self._spatial_rmse(model, self.obs_data).expand_dims(member=[0])
                    acc_t = [self._spatial_acc(model.isel(time=t), self.obs_data.isel(time=t)) for t in range(model.sizes['time'])]
                    acc_all = xr.concat(acc_t, dim='time').expand_dims(member=[0])

                rmse_all.to_dataset(name="rmse").to_netcdf(rmse_file)
                acc_all.to_dataset(name="acc").to_netcdf(acc_file)

            rmse_dict[exp] = {
                'mean': rmse_all.mean(dim='member'),
                'std': rmse_all.std(dim='member')
            }
            acc_dict[exp] = {
                'mean': acc_all.mean(dim='member'),
                'std': acc_all.std(dim='member')
            }

        # === Plot RMSE for all experiments ===
        plt.figure(figsize=(12, 5))
        for i, (exp, stat) in enumerate(rmse_dict.items()):
            color = plt.cm.tab10(i % 10)
            plt.plot(time, stat['mean'], label=f"{exp} Mean", color=color)
            plt.fill_between(time, stat['mean'] - stat['std'], stat['mean'] + stat['std'], color=color, alpha=0.2)
        plt.title(f"Time-Evolving RMSE in {self.region_alias}", fontsize=fontz + 2)
        plt.xlabel("Forecast Day", fontsize=fontz)
        plt.ylabel(f"RMSE [{self.unit}]", fontsize=fontz)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        rmse_plot = os.path.join(self.fig_dir, "fig_spatial_rmse_ts.pdf")
        plt.savefig(rmse_plot)
        plt.close()

        # === Plot ACC for all experiments ===
        plt.figure(figsize=(12, 5))
        for i, (exp, stat) in enumerate(acc_dict.items()):
            color = plt.cm.tab10(i % 10)
            plt.plot(time, stat['mean'], label=f"{exp} Mean", color=color)
            plt.fill_between(time, stat['mean'] - stat['std'], stat['mean'] + stat['std'], color=color, alpha=0.2)
        plt.title(f"Time-Evolving ACC in {self.region_alias}", fontsize=fontz + 2)
        plt.xlabel("Forecast Day", fontsize=fontz)
        plt.ylabel("ACC", fontsize=fontz)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        acc_plot = os.path.join(self.output_dir, "fig_spatial_acc_ts.pdf")
        plt.savefig(acc_plot)
        plt.close()

        return rmse_plot, acc_plot

    def run_all(self, force=False):
        print(f"[INFO] Running S2S predictability analysis for {self.region_alias}")
        self.read_data()
        metrics = self.compute_metrics()
        for exp, m in metrics.items():
            print(f"{exp}: " + ", ".join(f"{k}={v:.2f}" for k, v in m.items()))
        rmse_plot, acc_plot = self.plot_time_evolving_metrics(force=force)
        print(f"[INFO] Combined RMSE plot saved to: {rmse_plot}")
        print(f"[INFO] Combined ACC plot saved to:  {acc_plot}")
        return {
            "rmse_plot": rmse_plot,
            "acc_plot": acc_plot
        }


In [5]:
if __name__ == "__main__":
    # === Setup ===
    #top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    #out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    #fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    obsnam = "IMERG"
    var_dict = {
        'PRECT': {'unit': 'mm/day', 'alias': 'PRECT', 'min': 0, "max": 12}
    }
    frequency = "daily"
    model_list = ['CAPTEN10_15day', 
                  'DARTEN20_15day', 
                  'DARTEN40_15day']

    # === Region aliases for figure names ===
    regnam_dict = {
        "global": "Global",
        "CONUS": "Continental US",
        "NHMidLat": "Mid-Latitude (NH)",
        "Tropics": "Tropics"
    }

    for regnam, region_alias in regnam_dict.items():
        print(f"\n=== Processing region: {regnam} ===")

        analyzer = S2SPredictabilityAnalyzer(
            base_path=top_path,
            exp_base=exp_base,
            output_dir=out_path,
            fig_dir=fig_path,
            region=regnam,
            region_alias=region_alias,
            var_dict=var_dict,
            model_list=model_list,
            obsname=obsnam,
            frequency=frequency,
            component="atm"
        )
        analyzer.run_all()


=== Processing region: global ===
[INFO] Running S2S predictability analysis for Global


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packa

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packa

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packa

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


CAPTEN10_15day: RMSE_mean=10.46, RMSE_std=0.05, CORR_mean=0.02, CORR_std=0.01
DARTEN20_15day: RMSE_mean=10.53, RMSE_std=0.06, CORR_mean=0.02, CORR_std=0.01
DARTEN40_15day: RMSE_mean=10.53, RMSE_std=0.05, CORR_mean=0.02, CORR_std=0.01
[INFO] Skipping computation for CAPTEN10_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN20_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN40_15day, files exist. Loading from file.
[INFO] Combined RMSE plot saved to: ./fig_spatial_rmse_ts.pdf
[INFO] Combined ACC plot saved to:  ./data/fig_spatial_acc_ts.pdf

=== Processing region: CONUS ===
[INFO] Running S2S predictability analysis for Continental US
CAPTEN10_15day: RMSE_mean=6.06, RMSE_std=0.21, CORR_mean=0.03, CORR_std=0.03
DARTEN20_15day: RMSE_mean=6.28, RMSE_std=0.37, CORR_mean=0.01, CORR_std=0.04
DARTEN40_15day: RMSE_mean=6.25, RMSE_std=0.26, CORR_mean=0.02, CORR_std=0.04
[INFO] Skipping computation for CAPTEN10_15day, files exist. Loading fr

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


CAPTEN10_15day: RMSE_mean=6.65, RMSE_std=0.13, CORR_mean=0.01, CORR_std=0.01
DARTEN20_15day: RMSE_mean=6.63, RMSE_std=0.10, CORR_mean=0.00, CORR_std=0.01
DARTEN40_15day: RMSE_mean=6.77, RMSE_std=0.11, CORR_mean=0.01, CORR_std=0.01
[INFO] Skipping computation for CAPTEN10_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN20_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN40_15day, files exist. Loading from file.
[INFO] Combined RMSE plot saved to: ./fig_spatial_rmse_ts.pdf
[INFO] Combined ACC plot saved to:  ./data/fig_spatial_acc_ts.pdf

=== Processing region: Tropics ===
[INFO] Running S2S predictability analysis for Tropics


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/_task_spec.py:755: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


CAPTEN10_15day: RMSE_mean=10.63, RMSE_std=0.31, CORR_mean=0.06, CORR_std=0.02
DARTEN20_15day: RMSE_mean=10.85, RMSE_std=0.06, CORR_mean=0.04, CORR_std=0.02
DARTEN40_15day: RMSE_mean=10.81, RMSE_std=0.06, CORR_mean=0.04, CORR_std=0.01
[INFO] Skipping computation for CAPTEN10_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN20_15day, files exist. Loading from file.
[INFO] Skipping computation for DARTEN40_15day, files exist. Loading from file.
[INFO] Combined RMSE plot saved to: ./fig_spatial_rmse_ts.pdf
[INFO] Combined ACC plot saved to:  ./data/fig_spatial_acc_ts.pdf
